In [4]:
import csv
import gzip
import os

def split_and_compress_csv(file_path, max_size_mb=100):
    # Convert MB to bytes
    max_bytes = max_size_mb * 1024 * 1024
    
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    output_dir = os.path.dirname(file_path)
    if not output_dir:
        output_dir = '.'
        
    part_num = 1
    current_out_file = None
    current_writer = None
    current_size = 0
    header = None
    
    with open(file_path, mode='r', encoding='utf-8') as infile:
        reader = csv.reader(infile)
        header = next(reader) # Save the header
        
        for row in reader:
            # Estimate row size (len + 1 for newline)
            row_str = ','.join(row) + '\n'
            row_bytes = len(row_str.encode('utf-8'))
            
            # Create a new file if needed
            if current_writer is None or current_size + row_bytes > max_bytes:
                if current_out_file:
                    current_out_file.close()
                    
                part_path = os.path.join(output_dir, f"{base_name}_part{part_num}.csv.gz")
                current_out_file = gzip.open(part_path, mode='wt', encoding='utf-8', newline='')
                current_writer = csv.writer(current_out_file)
                
                # Write header to new part
                current_writer.writerow(header)
                current_size = sum(len(h.encode('utf-8')) + 1 for h in header)
                part_num += 1
                
            # Write the row
            current_writer.writerow(row)
            current_size += row_bytes
            
    if current_out_file:
        current_out_file.close()

# --- How to run it ---
# Replace 'large_file.csv' with the path to your file
# split_and_compress_csv('large_file.csv', max_size_mb=100)


In [5]:
file_path = '../data/DOB_Job_Application_Filings_20260613.csv'

In [6]:
split_and_compress_csv(file_path, max_size_mb=50)